# Data Preparation — Building the Fine-Tuning Dataset

## Goal
Prepare a domain-specific dataset for fine-tuning DistilGPT-2 on ML text.

## What We Need
- Text in the target domain (ML/NLP)
- Clean, well-formatted sentences
- Enough data to shift the model's distribution

## Data Sources
1. Manually written ML explanations
2. Abstracts from papers we already loaded
3. Key concepts from our own project notebooks

In [1]:
import json
import re
from pathlib import Path

## 1. Create ML Training Text
Domain-specific sentences about NLP, transformers, and ML concepts.

In [2]:
# ML/NLP training text — domain-specific sentences
training_texts = [
    # Transformers and Attention
    "The transformer architecture uses self-attention mechanisms to process sequences in parallel.",
    "Self-attention allows each token to attend to all other tokens in the sequence.",
    "Multi-head attention runs several attention functions simultaneously and concatenates the results.",
    "Positional encoding adds position information to token embeddings since transformers have no recurrence.",
    "The encoder processes the input sequence and the decoder generates the output sequence.",
    "BERT uses bidirectional attention to understand context from both left and right directions.",
    "GPT uses causal attention where each token can only attend to previous tokens.",
    "The attention score is computed as the dot product of query and key vectors divided by the square root of the dimension.",
    "Residual connections help gradients flow through deep transformer networks during training.",
    "Layer normalization stabilizes training by normalizing activations within each layer.",

    # Embeddings and Representations
    "Word embeddings map tokens to dense vectors where similar words are close in vector space.",
    "Sentence embeddings represent entire sentences as single vectors for downstream tasks.",
    "SBERT fine-tunes BERT using siamese networks to produce semantically meaningful sentence embeddings.",
    "Cosine similarity measures the angle between two vectors and is commonly used for text similarity.",
    "TF-IDF weighs words by their frequency in a document relative to their frequency in the corpus.",
    "Mean pooling averages all token embeddings to produce a single sentence representation.",
    "The CLS token in BERT is used as a summary representation for classification tasks.",

    # RAG and Retrieval
    "Retrieval-Augmented Generation combines a retriever and a generator to answer questions from documents.",
    "FAISS enables efficient similarity search over large collections of dense vectors.",
    "Chunking strategy determines how documents are split before embedding and indexing.",
    "A chunk that cuts mid-sentence produces a noisy embedding that retrieves poorly.",
    "Sentence-based chunking preserves semantic units and produces cleaner embeddings.",
    "The grounding score measures how much the generated answer is supported by the retrieved context.",
    "Hallucination occurs when a model generates facts not present in the provided context.",
    "Top-k retrieval returns the k most similar chunks to the query vector.",

    # Training and Fine-Tuning
    "Fine-tuning adapts a pre-trained model to a specific domain using task-specific data.",
    "Language model fine-tuning trains the model to predict the next token on domain text.",
    "Perplexity measures how surprised a model is by a text — lower perplexity means better understanding.",
    "Cross-entropy loss measures the difference between predicted and actual token distributions.",
    "Learning rate controls how much the weights change during each training step.",
    "Gradient descent updates weights in the direction that reduces the loss function.",
    "Overfitting occurs when a model learns the training data too well and fails to generalize.",
    "Early stopping halts training when validation loss stops improving to prevent overfitting.",

    # NLP Tasks
    "Named entity recognition identifies and classifies named entities in text.",
    "Sentiment analysis classifies text as positive, negative, or neutral.",
    "Text classification assigns predefined categories to input documents.",
    "Machine translation converts text from one language to another.",
    "Summarization produces a shorter version of a document that preserves key information.",
    "Question answering systems find answers to questions in a given context.",
    "Semantic similarity measures how close two sentences are in meaning.",
    "Information retrieval finds relevant documents from a large corpus given a query.",
]

print(f"Total training sentences: {len(training_texts)}")
print(f"Total words: {sum(len(t.split()) for t in training_texts)}")
print(f"\nSample sentences:")
for t in training_texts[:3]:
    print(f"  - {t}")

Total training sentences: 41
Total words: 506

Sample sentences:
  - The transformer architecture uses self-attention mechanisms to process sequences in parallel.
  - Self-attention allows each token to attend to all other tokens in the sequence.
  - Multi-head attention runs several attention functions simultaneously and concatenates the results.


## 2. Process and Save Dataset

In [4]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

# Combine all texts into one document
full_text = "\n".join(training_texts)

# Tokenize and check lenght
tokens = tokenizer(full_text, return_tensors="pt")
print(f"Total tokens: {tokens.input_ids.shape[1]}")

# Save raw text
Path("../data/raw").mkdir(parents=True, exist_ok=True)
with open("../data/raw/ml_training_text.txt", "w") as f:
    f.write(full_text)

# Save processed - one sentence per line
Path("../data/processed").mkdir(parents=True, exist_ok=True)
with open("../data/processed/ml_sentences.txt", "w") as f:
    for text in training_texts:
        f.write(text + "\n")

print(f"Saved {len(training_texts)} sentences to data/processed/ml_senteces.txt")
print(f"\nToken lenght distribution:")
lenghts = [len(tokenizer(t).input_ids) for t in training_texts]
print(f" Min: {min(lenghts)}")
print(f" Max: {max(lenghts)}")
print(f" Avg: {sum(lenghts)/len(lenghts):.1f}")

Total tokens: 698
Saved 41 sentences to data/processed/ml_senteces.txt

Token lenght distribution:
 Min: 10
 Max: 24
 Avg: 16.0


## 3. Key Observations

| Metric | Value |
|--------|-------|
| Total sentences | 41 |
| Total tokens | 698 |
| Avg tokens per sentence | 16 |
| Min tokens | 10 |
| Max tokens | 24 |

## Key Insight
698 tokens is a very small dataset for fine-tuning.
In production, you would use thousands or millions of tokens.

For this experiment, small data is intentional:
- We want to see the model ch